# FA validation: artificial B0 versus acquired reverse-PE B0

Compare **one subject and session** processed with the same DWI, tensor-fitting settings,
and downstream steps, changing only the susceptibility-correction method.
The acquired reverse-PE result is the comparison reference, not ground truth.

1. Edit the paths below, then run all cells.
2. Prefer native-space FA maps (`.nii` or `.nii.gz`) and a common, independently defined
   brain or white-matter mask. A mask must be a binary NIfTI on the FA grid;
   convert MRtrix masks with `mrconvert mask.mif mask.nii.gz` if needed.
3. Images must share a voxel grid **and be anatomically aligned**. Matching headers
   alone do not establish alignment. Inspect the images; this notebook does no registration
   or resampling because interpolation itself can change FA distributions.

Dependencies: `numpy`, `nibabel`, `matplotlib`, `scipy` in your Jupyter kernel.
If needed, run `%pip install numpy nibabel matplotlib scipy` in a separate cell.


In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, wasserstein_distance

# Pipeline examples; replace these with paths to your own subject/session.
SESSION_DIR = Path('/path/to/bids/sub-001/ses-1')
SUBJECT = 'sub-001'
FA_ORIGINAL = SESSION_DIR / 'dwi/preprocessing/reversepe' / f'{SUBJECT}_model-dti_FA.nii.gz'
FA_ARTIFICIAL = SESSION_DIR / 'dwi/preprocessing/synb0' / f'{SUBJECT}_model-dti_FA.nii.gz'

# Recommended: a common brain/WM ROI, defined independently of these FA values.
MASK_PATH = None  # e.g. Path('/path/to/common_wm_mask.nii.gz')
N_BINS = 60

def load_volume(path):
    img = nib.load(str(Path(path).expanduser()))
    if len(img.shape) != 3:
        raise ValueError(f'Expected a 3D NIfTI volume: {path}, got {img.shape}')
    return img, img.get_fdata(dtype=np.float32)


def check_grid(img, reference, name):
    if img.shape != reference.shape or not np.allclose(
        img.affine, reference.affine, rtol=0, atol=1e-4
    ):
        raise ValueError(
            f'{name} does not share the original FA grid. Check alignment and '
            'prepare aligned images/masks before comparing; no automatic resampling.'
        )


## Load → check geometry → select a common set of voxels

With a mask, FA = 0 is retained because it is a valid isotropic value. Without a mask,
use the intersection of nonzero support in both images to remove background. This
fallback also removes true zero-FA tissue and can hide differences in coverage;
use an explicit common mask for the final comparison. Do not threshold each FA map
separately to define white matter: that changes the populations being compared.

Nonfinite and out-of-range values are counted, then excluded **jointly** from both
images without clipping. Investigate substantial exclusions before interpreting results.


In [ ]:
original_img, fa_original = load_volume(FA_ORIGINAL)
artificial_img, fa_artificial = load_volume(FA_ARTIFICIAL)
check_grid(artificial_img, original_img, 'Artificial FA')

if MASK_PATH is not None:
    mask_img, mask_data = load_volume(MASK_PATH)
    check_grid(mask_img, original_img, 'Mask')
    if not np.all(np.isfinite(mask_data)) or not np.all(np.isin(mask_data, [0, 1])):
        raise ValueError('Supply a finite binary mask containing only 0 and 1.')
    roi = mask_data == 1
    print('ROI: supplied common mask (zero FA retained).')
else:
    roi = (fa_original != 0) & (fa_artificial != 0)
    print('Exploratory ROI: intersection of nonzero support; true zero FA is excluded.')
    print(f'Original nonzero support: {np.count_nonzero(np.isfinite(fa_original) & (fa_original != 0)):,}')
    print(f'Artificial nonzero support: {np.count_nonzero(np.isfinite(fa_artificial) & (fa_artificial != 0)):,}')

for label, fa in [('Original', fa_original), ('Artificial', fa_artificial)]:
    nonfinite = roi & ~np.isfinite(fa)
    out_of_range = roi & np.isfinite(fa) & ((fa < 0) | (fa > 1))
    print(f'{label}: {nonfinite.sum():,} nonfinite, {out_of_range.sum():,} outside [0, 1] in ROI')

# Use exactly the same locations for both distributions and paired differences.
valid_original = np.isfinite(fa_original) & (fa_original >= 0) & (fa_original <= 1)
valid_artificial = np.isfinite(fa_artificial) & (fa_artificial >= 0) & (fa_artificial <= 1)
common_mask = roi & valid_original & valid_artificial
n_roi = int(roi.sum())
n_valid = int(common_mask.sum())
if n_valid == 0:
    raise ValueError('No valid common voxels remain; check the input images and mask.')

x = fa_original[common_mask].astype(float)
y = fa_artificial[common_mask].astype(float)
print(f'Grid: {original_img.shape}; voxel sizes (mm): {nib.affines.voxel_sizes(original_img.affine)}')
print(f'ROI voxels: {n_roi:,}; retained: {n_valid:,}; jointly excluded: {n_roi - n_valid:,}')
print(f'Original:   {Path(FA_ORIGINAL).expanduser().resolve()}')
print(f'Artificial: {Path(FA_ARTIFICIAL).expanduser().resolve()}')


In [ ]:
# Summarize the same voxel population in each reconstruction.
print(f"{'Method':<15} {'Mean':>9} {'SD':>9} {'P05':>9} {'Median':>9} {'P95':>9}")
for label, values in [('Original', x), ('Artificial', y)]:
    p05, median, p95 = np.percentile(values, [5, 50, 95])
    print(f'{label:<15} {values.mean():9.4f} {values.std(ddof=0):9.4f} '
          f'{p05:9.4f} {median:9.4f} {p95:9.4f}')

# Wasserstein measures distribution displacement in FA units; KS is the largest
# absolute difference between the two cumulative distribution functions.
wasserstein = wasserstein_distance(x, y)
ks_distance = ks_2samp(x, y).statistic

# Paired errors detect spatial disagreement even when histograms match.
delta = y - x  # Positive means higher FA with the artificial B0.
bias = delta.mean()
mae = np.abs(delta).mean()
rmse = np.sqrt(np.mean(delta**2))
print(f'\nWasserstein distance (FA units): {wasserstein:.5f}')
print(f'KS distance (0–1):              {ks_distance:.5f}')
print(f'Mean paired difference (FA):    {bias:+.5f}')
print(f'Mean absolute error (FA):       {mae:.5f}')
print(f'Root mean squared error (FA):   {rmse:.5f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
bins = np.linspace(0, 1, N_BINS + 1)
for label, values, color in [('Original reverse-PE', x, 'C0'),
                             ('Artificial B0', y, 'C1')]:
    axes[0].hist(values, bins=bins, density=True, histtype='step',
                 linewidth=2, color=color, label=label)
    sorted_values = np.sort(values)
    # Include the lower step endpoint so ties and singleton inputs are explicit.
    axes[1].step(np.r_[sorted_values[0], sorted_values],
                 np.r_[0, np.arange(1, values.size + 1) / values.size],
                 where='post', color=color, label=label)

axes[0].set(title='FA distribution: common voxels', xlabel='FA', ylabel='Probability density')
axes[1].set(title='Empirical cumulative distribution', xlabel='FA', ylabel='Fraction of voxels ≤ FA')
for ax in axes:
    ax.set_xlim(0, 1)
    ax.legend()
    ax.grid(alpha=0.2)
plt.show()


In [ ]:
# Show the axial slice with the largest number of selected voxels.
# These are array slices, not a radiological orientation display.
z = int(np.argmax(common_mask.sum(axis=(0, 1))))
difference_map = np.full(fa_original.shape, np.nan, dtype=np.float32)
difference_map[common_mask] = delta
color_limit = max(float(np.percentile(np.abs(delta), 99)), 0.01)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, fa, title in zip(axes[:2], [fa_original, fa_artificial],
                         ['Original reverse-PE', 'Artificial B0']):
    plane = np.where(common_mask[:, :, z], fa[:, :, z], np.nan)
    fa_plot = ax.imshow(plane.T, origin='lower', cmap='gray', vmin=0, vmax=1)
    ax.set_title(title)
fig.colorbar(fa_plot, ax=list(axes[:2]), label='FA', shrink=0.7)
diff_plot = axes[2].imshow(difference_map[:, :, z].T, origin='lower',
                           cmap='RdBu_r', vmin=-color_limit, vmax=color_limit)
axes[2].set_title('Artificial − original')
fig.colorbar(diff_plot, ax=axes[2], label='FA difference', shrink=0.7, extend='both')
for ax in axes:
    ax.set_aspect('equal')
    ax.axis('off')
fig.suptitle(f'Common ROI, voxel slice z = {z}; difference colors saturate at ±{color_limit:.3f}')
plt.show()


## Interpretation

- Look for overlapping histograms and cumulative curves, small Wasserstein/KS distances,
  and small paired bias/MAE/RMSE. Zero indicates exact agreement for these metrics.
- Positive bias means the artificial-B0 result has higher FA on average. A small bias can
  coexist with large errors because positive and negative differences cancel.
- Similar whole-brain distributions do not establish anatomical equivalence. Inspect
  spatial differences, especially near susceptibility-prone regions; repeat in independent
  white-matter or anatomical ROIs when relevant.
- No automatic pass/fail threshold is imposed: define acceptable FA differences for your
  scientific use case before drawing conclusions. Voxel counts are not independent sample
  sizes because neighboring voxels are spatially correlated; a voxelwise KS p-value is not
  used as evidence of equivalence.
- For multiple subjects, run the same comparison with consistent masking for each matched
  session and summarize subject-level metrics rather than pooling all voxels.


## Batch comparison: all subjects and sessions in `doc_data/`

Run the **imports/settings cell** above, then this cell; the single-subject cells are
optional. Each session needs the saved `reversepe` and `synb0` FA maps. Outputs go to
repository-root `plots/`: one PNG per successful session and `fa_distances.txt`
(a tab-separated table containing distances, descriptive statistics, voxel counts,
and the status/reason for skipped sessions). Rerunning replaces outputs with the same
names; the log describes the current run, while older PNGs may remain in the folder.

By default, use the same exploratory intersection of nonzero support as above.
Set `BATCH_MASK_RELATIVE` to a common binary mask path relative to **each session**
for a masked analysis. The single-subject `MASK_PATH` is not reused across subjects.


In [2]:
import csv
from datetime import datetime, timezone

# Find the repository when Jupyter starts at its root or inside scripts/.
REPO_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'doc_data').is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError('Start Jupyter inside the repository, or set REPO_DIR explicitly.')
DATA_DIR = REPO_DIR / 'doc_data'
PLOTS_DIR = REPO_DIR / 'plots'
BATCH_MASK_RELATIVE = None  # e.g. 'dwi/common_wm_mask.nii.gz'; relative to each session

subjects = sorted(p for p in DATA_DIR.glob('sub-*') if p.is_dir())
if not subjects:
    raise FileNotFoundError(f'No sub-* directories found in {DATA_DIR}')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
log_path = PLOTS_DIR / 'fa_distances.txt'
fields = ['subject', 'session', 'status', 'reason', 'original_path', 'artificial_path',
          'mask_path', 'roi_voxels', 'valid_voxels', 'excluded_voxels',
          'original_nonfinite', 'artificial_nonfinite',
          'original_out_of_range', 'artificial_out_of_range',
          'wasserstein', 'ks_distance', 'bias_artificial_minus_original', 'mae', 'rmse',
          'original_mean', 'artificial_mean', 'original_sd', 'artificial_sd',
          'original_median', 'artificial_median', 'plot_path']
completed = skipped = 0

with log_path.open('w', newline='', encoding='utf-8') as log:
    log.write(f'# FA comparison; UTC: {datetime.now(timezone.utc).isoformat()}\n')
    log.write('# Reference: acquired reversepe; comparison: artificial synb0.\n')
    log.write('# Distances/errors in FA units except KS (0–1); SD uses ddof=0.\n')
    log.write('# ROI: ' + ('session-specific common binary mask' if BATCH_MASK_RELATIVE
                            else 'nonzero-support intersection; true zero FA excluded') + '\n')
    writer = csv.DictWriter(log, fieldnames=fields, delimiter='\t')
    writer.writeheader()
    for subject_dir in subjects:
        sessions = sorted(p for p in subject_dir.glob('ses-*') if p.is_dir())
        if not sessions:
            writer.writerow(dict(subject=subject_dir.name, status='SKIPPED', reason='No ses-* directories'))
            skipped += 1
        for session_dir in sessions:
            subject, session = subject_dir.name, session_dir.name
            base = session_dir / 'dwi/preprocessing'
            original_path = base / 'reversepe' / f'{subject}_model-dti_FA.nii.gz'
            artificial_path = base / 'synb0' / f'{subject}_model-dti_FA.nii.gz'
            mask_path = session_dir / BATCH_MASK_RELATIVE if BATCH_MASK_RELATIVE else None
            row = dict(subject=subject, session=session, original_path=str(original_path),
                       artificial_path=str(artificial_path), mask_path=str(mask_path or ''))
            batch_fig = None
            try:
                ref_img, ref = load_volume(original_path)
                syn_img, syn = load_volume(artificial_path)
                check_grid(syn_img, ref_img, 'Artificial FA')
                if mask_path is None:
                    roi_batch = (ref != 0) & (syn != 0)
                else:
                    roi_img, roi_values = load_volume(mask_path)
                    check_grid(roi_img, ref_img, 'Mask')
                    if not np.all(np.isfinite(roi_values)) or not np.all(np.isin(roi_values, [0, 1])):
                        raise ValueError('Common mask must contain only finite 0 and 1 values.')
                    roi_batch = roi_values == 1
                finite_ref, finite_syn = np.isfinite(ref), np.isfinite(syn)
                range_ref = finite_ref & ((ref < 0) | (ref > 1))
                range_syn = finite_syn & ((syn < 0) | (syn > 1))
                selected = roi_batch & finite_ref & finite_syn & ~range_ref & ~range_syn
                n = int(selected.sum())
                row.update(roi_voxels=int(roi_batch.sum()), valid_voxels=n,
                           excluded_voxels=int(roi_batch.sum()) - n,
                           original_nonfinite=int((roi_batch & ~finite_ref).sum()),
                           artificial_nonfinite=int((roi_batch & ~finite_syn).sum()),
                           original_out_of_range=int((roi_batch & range_ref).sum()),
                           artificial_out_of_range=int((roi_batch & range_syn).sum()))
                if n == 0:
                    raise ValueError('No valid common voxels remain.')
                ref_values = ref[selected].astype(float)
                syn_values = syn[selected].astype(float)
                differences = syn_values - ref_values
                row.update(wasserstein=float(wasserstein_distance(ref_values, syn_values)),
                           ks_distance=float(ks_2samp(ref_values, syn_values).statistic),
                           bias_artificial_minus_original=float(differences.mean()),
                           mae=float(np.abs(differences).mean()),
                           rmse=float(np.sqrt(np.mean(differences**2))),
                           original_mean=float(ref_values.mean()), artificial_mean=float(syn_values.mean()),
                           original_sd=float(ref_values.std()), artificial_sd=float(syn_values.std()),
                           original_median=float(np.median(ref_values)),
                           artificial_median=float(np.median(syn_values)))

                batch_fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
                for label, values, color in [('Original reverse-PE', ref_values, 'C0'),
                                             ('Artificial B0', syn_values, 'C1')]:
                    axes[0, 0].hist(values, bins=np.linspace(0, 1, N_BINS + 1), density=True,
                                    histtype='step', linewidth=2, color=color, label=label)
                    ordered = np.sort(values)
                    axes[0, 1].step(np.r_[ordered[0], ordered], np.r_[0, np.arange(1, n + 1) / n],
                                    where='post', color=color, label=label)
                axes[0, 0].set(title='FA distributions', xlabel='FA', ylabel='Probability density', xlim=(0, 1))
                axes[0, 1].set(title='Cumulative distributions', xlabel='FA', ylabel='Fraction ≤ FA', xlim=(0, 1))
                axes[0, 2].hist(differences, bins=N_BINS, color='0.4', density=True)
                axes[0, 2].axvline(0, color='black', linestyle='--')
                axes[0, 2].set(title='Paired FA differences', xlabel='Artificial − original', ylabel='Probability density')
                for ax in axes[0, :2]:
                    ax.legend()
                z_batch = int(np.argmax(selected.sum(axis=(0, 1))))
                for ax, data, label in zip(axes[1, :2], [ref, syn], ['Original reverse-PE', 'Artificial B0']):
                    plane = np.where(selected[:, :, z_batch], data[:, :, z_batch], np.nan)
                    fa_image = ax.imshow(plane.T, origin='lower', cmap='gray', vmin=0, vmax=1)
                    ax.set_title(label)
                batch_fig.colorbar(fa_image, ax=list(axes[1, :2]), label='FA', shrink=0.7)
                diff_plane = np.where(selected[:, :, z_batch], syn[:, :, z_batch] - ref[:, :, z_batch], np.nan)
                limit = max(float(np.percentile(np.abs(differences), 99)), 0.01)
                diff_image = axes[1, 2].imshow(diff_plane.T, origin='lower', cmap='RdBu_r', vmin=-limit, vmax=limit)
                axes[1, 2].set_title(f'Artificial − original (±{limit:.3f})')
                batch_fig.colorbar(diff_image, ax=axes[1, 2], label='FA difference', shrink=0.7, extend='both')
                for ax in axes[1]:
                    ax.axis('off')
                batch_fig.suptitle(f'{subject} / {session} | {n:,} voxels | array slice z={z_batch}\n'
                                  f'Wasserstein={row["wasserstein"]:.5f}, KS={row["ks_distance"]:.5f}, '
                                  f'bias={row["bias_artificial_minus_original"]:+.5f}, '
                                  f'MAE={row["mae"]:.5f}, RMSE={row["rmse"]:.5f}')
                plot_path = PLOTS_DIR / f'{subject}_{session}_fa_comparison.png'
                batch_fig.savefig(plot_path, dpi=150)
                row.update(status='OK', plot_path=str(plot_path))
                completed += 1
                print(f'Saved {subject}/{session}')
            except (OSError, ValueError, nib.filebasedimages.ImageFileError) as exc:
                row.update(status='SKIPPED', reason=' '.join(str(exc).split()))
                skipped += 1
                print(f'Skipped {subject}/{session}: {row["reason"]}')
            finally:
                if batch_fig is not None:
                    plt.close(batch_fig)
            writer.writerow(row)
            log.flush()

print(f'Completed: {completed}; skipped: {skipped}. Plots: {PLOTS_DIR}\nMetrics log: {log_path}')


Saved sub-BG391/ses-1
Skipped sub-BG391/ses-2: No such file or no access: '/network/iss/home/ivan.mindlin/Repos/Tractography_pipeline/doc_data/sub-BG391/ses-2/dwi/preprocessing/reversepe/sub-BG391_model-dti_FA.nii.gz'
Saved sub-FA427/ses-1
Saved sub-FT410/ses-1
Skipped sub-GL417/ses-1: No such file or no access: '/network/iss/home/ivan.mindlin/Repos/Tractography_pipeline/doc_data/sub-GL417/ses-1/dwi/preprocessing/reversepe/sub-GL417_model-dti_FA.nii.gz'
Skipped sub-GL417/ses-2: No such file or no access: '/network/iss/home/ivan.mindlin/Repos/Tractography_pipeline/doc_data/sub-GL417/ses-2/dwi/preprocessing/reversepe/sub-GL417_model-dti_FA.nii.gz'
Saved sub-LE468/ses-1
Skipped sub-MD414/ses-1: No such file or no access: '/network/iss/home/ivan.mindlin/Repos/Tractography_pipeline/doc_data/sub-MD414/ses-1/dwi/preprocessing/reversepe/sub-MD414_model-dti_FA.nii.gz'
Skipped sub-NO412/ses-1: No such file or no access: '/network/iss/home/ivan.mindlin/Repos/Tractography_pipeline/doc_data/sub-NO4